# Clasificador de Patologías Estructurales en Hormigón Armado

Modelo XGBoost entrenado con dataset sintético de observaciones de campo.

**Pipeline:** Carga → Preprocesamiento → Split → Entrenamiento base

In [ ]:
# ============================================================
# Celda 1: Importar librerías necesarias
# Se importan todas las herramientas que usaremos en el notebook.
# ============================================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

print("Librerías cargadas correctamente.")

In [ ]:
# ============================================================
# Celda 2: Cargar el dataset sintético
# El CSV fue generado por generar_dataset_sintetico.py con
# 2216 observaciones de campo simuladas (96 patologías).
# ============================================================

df = pd.read_csv("dataset_patologias_sintetico.csv")

print(f"Dimensiones del dataset: {df.shape}")
print(f"Patologías únicas: {df['defecto_numero'].nunique()}")
print(f"\nPrimeras 5 filas:")
df.head()

In [ ]:
# ============================================================
# Celda 3: Separar features (X) y target (y)
# Todas las columnas son categóricas excepto defecto_numero,
# que es la variable objetivo (1 a 96).
# XGBoost necesita clases desde 0, así que restamos 1.
# Guardamos el mapeo para reconvertir después.
# ============================================================

X = df.drop(columns=["defecto_numero"])
y = df["defecto_numero"] - 1  # Convertir a 0-based (0 a 95)

columnas_categoricas = X.columns.tolist()

print(f"Features ({len(columnas_categoricas)}): {columnas_categoricas}")
print(f"Target: defecto_numero convertido a 0-based (clases {y.min()} a {y.max()})")

In [ ]:
# ============================================================
# Celda 4: Preprocesamiento con ColumnTransformer
# Se usa OneHotEncoder dentro de un ColumnTransformer para
# convertir todas las variables categóricas a numéricas.
# handle_unknown='ignore' permite que en predicción futura
# no falle si aparece un valor no visto en entrenamiento.
# ============================================================

preprocesador = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), columnas_categoricas)
    ]
)

print("Preprocesador definido: OneHotEncoder para todas las features categóricas.")
print(f"Se transformarán {len(columnas_categoricas)} columnas.")

In [ ]:
# ============================================================
# Celda 5: Split train/test 80/20 con stratify
# stratify=y asegura que cada clase tenga la misma proporción
# en train y test, importante con 96 clases desbalanceadas.
# random_state=42 para reproducibilidad.
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Train: {X_train.shape[0]} muestras")
print(f"Test:  {X_test.shape[0]} muestras")
print(f"\nClases en train: {y_train.nunique()} | Clases en test: {y_test.nunique()}")
print(f"\nDistribución en train (resumen):")
print(y_train.value_counts().describe())

In [ ]:
# ============================================================
# Celda 6: Pipeline completo (preprocesamiento + XGBoost base)
# Se encapsula todo en un Pipeline de sklearn para que el
# preprocesamiento y el modelo sean un solo objeto reutilizable.
# Modelo base con parámetros razonables sin optimizar aún.
# No se fija num_class: XGBoost lo infiere de y automáticamente.
# ============================================================

modelo_base = Pipeline(steps=[
    ("preprocesador", preprocesador),
    ("clasificador", XGBClassifier(
        n_estimators=200,
        eta=0.1,
        max_depth=6,
        gamma=0.5,
        objective="multi:softprob",
        eval_metric="mlogloss",
        random_state=42,
        verbosity=0
    ))
])

print("Pipeline definido:")
print(modelo_base)

In [ ]:
# ============================================================
# Celda 7: Entrenar el modelo base
# Entrenamos con los datos de train y evaluamos en train y test
# para tener una primera línea base antes de GridSearch.
# ============================================================

modelo_base.fit(X_train, y_train)

# Predicciones en train y test
y_pred_train = modelo_base.predict(X_train)
y_pred_test = modelo_base.predict(X_test)

acc_train = accuracy_score(y_train, y_pred_train)
acc_test = accuracy_score(y_test, y_pred_test)

print(f"Accuracy en TRAIN: {acc_train:.4f}")
print(f"Accuracy en TEST:  {acc_test:.4f}")
print(f"\nDiferencia (sobreajuste): {acc_train - acc_test:.4f}")
print("\n" + "="*50)
print("CLASSIFICATION REPORT (TEST):")
print("="*50)
print(classification_report(y_test, y_pred_test, zero_division=0))

## GridSearchCV: Búsqueda de hiperparámetros óptimos

Se prueban combinaciones de `n_estimators`, `eta`, `gamma` y `max_depth` con validación cruzada de 4 folds.  
Se evalúan 4 métricas y se reajusta por `f1_macro`.

In [ ]:
# ============================================================
# Celda 8: Definir grilla de hiperparámetros y GridSearchCV
# Se prueban 3×2×3×2 = 36 combinaciones con cv=4 folds.
# Se evalúan 4 métricas (accuracy, precision, recall, f1 macro)
# y se reajusta el modelo final por f1_macro.
# Los nombres de parámetros llevan prefijo "clasificador__"
# porque están dentro del Pipeline.
# ============================================================

from sklearn.model_selection import GridSearchCV

parametros_grilla = {
    "clasificador__n_estimators": [100, 200, 300],
    "clasificador__eta": [0.1, 0.4],
    "clasificador__gamma": [0.1, 0.5, 1],
    "clasificador__max_depth": [5, 10],
}

metricas_evaluacion = {
    "accuracy": "accuracy",
    "precision_macro": "precision_macro",
    "recall_macro": "recall_macro",
    "f1_macro": "f1_macro",
}

# Pipeline limpio para GridSearch (sin entrenar)
pipeline_grid = Pipeline(steps=[
    ("preprocesador", preprocesador),
    ("clasificador", XGBClassifier(
        objective="multi:softprob",
        eval_metric="mlogloss",
        random_state=42,
        verbosity=0
    ))
])

grid_search = GridSearchCV(
    estimator=pipeline_grid,
    param_grid=parametros_grilla,
    scoring=metricas_evaluacion,
    refit="f1_macro",
    cv=4,
    n_jobs=-1,
    verbose=1,
    return_train_score=True
)

n_combinaciones = 1
for v in parametros_grilla.values():
    n_combinaciones *= len(v)

print(f"Combinaciones a evaluar: {n_combinaciones}")
print(f"Folds por combinación: 4")
print(f"Total de fits: {n_combinaciones * 4}")
print(f"\nMétricas de evaluación: {list(metricas_evaluacion.keys())}")
print(f"Refit por: f1_macro")

In [ ]:
# ============================================================
# Celda 9: Ejecutar GridSearchCV
# Esto puede tardar varios minutos porque entrena 36 modelos
# (36 combinaciones × 4 folds = 144 fits).
# n_jobs=-1 usa todos los cores disponibles para paralelizar.
# ============================================================

print("Iniciando búsqueda de hiperparámetros...")
grid_search.fit(X_train, y_train)

print(f"\nMejor f1_macro en CV: {grid_search.best_score_:.4f}")
print(f"\nMejores hiperparámetros:")
for param, valor in grid_search.best_params_.items():
    print(f"  {param}: {valor}")

In [ ]:
# ============================================================
# Celda 10: Tabla resumen de resultados del GridSearch
# Ordenamos todas las combinaciones por f1_macro para ver
# cuáles fueron los mejores y peores modelos.
# ============================================================

resultados = pd.DataFrame(grid_search.cv_results_)

# Columnas de interés: parámetros + métricas medias de test
columnas_resumen = (
    [c for c in resultados.columns if c.startswith("param_")]
    + ["mean_test_accuracy", "mean_test_precision_macro",
       "mean_test_recall_macro", "mean_test_f1_macro", "rank_test_f1_macro"]
)

resumen = resultados[columnas_resumen].sort_values("rank_test_f1_macro")
print("Top 10 combinaciones por f1_macro:")
resumen.head(10)

In [ ]:
# ============================================================
# Celda 11: Evaluar sobreajuste del mejor modelo (GridSearch)
# Comparamos accuracy en train vs test del modelo reajustado
# para verificar si el GridSearch ayudó a reducir sobreajuste.
# ============================================================

mejor_modelo = grid_search.best_estimator_

y_pred_train_gs = mejor_modelo.predict(X_train)
y_pred_test_gs = mejor_modelo.predict(X_test)

acc_train_gs = accuracy_score(y_train, y_pred_train_gs)
acc_test_gs = accuracy_score(y_test, y_pred_test_gs)

print("=" * 50)
print("COMPARACIÓN: Modelo base vs Mejor modelo GridSearch")
print("=" * 50)
print(f"{'Métrica':<25} {'Base':>10} {'GridSearch':>12}")
print("-" * 50)
print(f"{'Accuracy Train':<25} {acc_train:.4f}{'':>4} {acc_train_gs:.4f}")
print(f"{'Accuracy Test':<25} {acc_test:.4f}{'':>4} {acc_test_gs:.4f}")
print(f"{'Sobreajuste (diff)':<25} {acc_train - acc_test:.4f}{'':>4} {acc_train_gs - acc_test_gs:.4f}")
print(f"\nMejor f1_macro en CV: {grid_search.best_score_:.4f}")

In [ ]:
# ============================================================
# Celda 12: Boxplots de robustez de los top 5 modelos
# Para cada uno de los 5 mejores modelos (por f1_macro),
# extraemos los scores de f1 en cada uno de los 4 folds.
# Esto muestra qué tan estable es el rendimiento del modelo
# entre distintos subconjuntos de datos (robustez).
# ============================================================

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

# Índices de los top 5 modelos por f1_macro
top5_indices = resultados.nsmallest(5, "rank_test_f1_macro").index

# Extraer los scores de f1_macro por fold para cada modelo top 5
datos_boxplot = []
etiquetas = []

for rank, idx in enumerate(top5_indices, 1):
    # Los scores individuales por fold están en split{i}_test_f1_macro
    scores_folds = [resultados.loc[idx, f"split{i}_test_f1_macro"] for i in range(4)]
    datos_boxplot.append(scores_folds)

    # Etiqueta con los parámetros del modelo
    params = resultados.loc[idx]
    etiqueta = (
        f"#{rank}\n"
        f"n={params['param_clasificador__n_estimators']}, "
        f"eta={params['param_clasificador__eta']}\n"
        f"gamma={params['param_clasificador__gamma']}, "
        f"depth={params['param_clasificador__max_depth']}"
    )
    etiquetas.append(etiqueta)

fig, ax = plt.subplots(figsize=(12, 6))
bp = ax.boxplot(datos_boxplot, labels=etiquetas, patch_artist=True, widths=0.5)

# Colores para los boxplots
colores = sns.color_palette("viridis", 5)
for patch, color in zip(bp["boxes"], colores):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_title("Robustez de los Top 5 Modelos (f1_macro por fold en CV)", fontsize=14, fontweight="bold")
ax.set_ylabel("f1_macro", fontsize=12)
ax.set_xlabel("Modelo (hiperparámetros)", fontsize=12)
ax.tick_params(axis="x", labelsize=9)

# Línea horizontal con la media del mejor modelo
media_mejor = resultados.loc[top5_indices[0], "mean_test_f1_macro"]
ax.axhline(y=media_mejor, color="red", linestyle="--", alpha=0.5,
           label=f"Media mejor modelo: {media_mejor:.4f}")
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig("boxplot_robustez_top5.png", dpi=150, bbox_inches="tight")
plt.show()

print("Gráfico guardado: boxplot_robustez_top5.png")